In [6]:
import os
from ultralytics import YOLO
import pandas as pd
from tqdm import tqdm

In [7]:
# Creating a list with the paths to each patient's folder.
print(os.getcwd())
DIR = '../data/detection/patients'
patient_folders = sorted([os.path.join(DIR, x, 'images') for x in os.listdir(DIR) if os.path.isdir(os.path.join(DIR, x))])
patient_folders

/home/lab/code/myelocare/datasets/MMDB/src


['../data/detection/patients/patient 01/images',
 '../data/detection/patients/patient 02/images',
 '../data/detection/patients/patient 03/images',
 '../data/detection/patients/patient 04/images',
 '../data/detection/patients/patient 05/images',
 '../data/detection/patients/patient 06/images',
 '../data/detection/patients/patient 07/images',
 '../data/detection/patients/patient 08/images',
 '../data/detection/patients/patient 09/images',
 '../data/detection/patients/patient 10/images']

In [35]:

model = YOLO('../../../../diffusion_model/synthetic_orignal_merged/5fold/runs/fold2/weights/best.pt') # Weights of our final model.
# model = YOLO('../../../../myelocare/pretrained_models/pmmdc_150epochs/best.pt')
CONF = 0.8 # Confidence threshold for our final model obtained during validation.

In [36]:
# Initialize an empty list to store the results for each patient.
results = []

# Iterate through the folders of each patient
for patient_folder in tqdm(patient_folders):

    # Get the list of image files within the current patient's folder.
    image_files = os.listdir(patient_folder)

    # Extract the patient's identifier from the folder path.
    patient = int(patient_folder.split('/')[-2].split(' ')[-1])

    # Initialize counters for plasma cells and non-plasma cells.
    plasma_cells = 0
    non_plasma_cells = 0
    
    # Iterate through each image file in the current patient's folder.
    for image_file in image_files:
        # Use the model to make predictions on the image.
        detections = model(os.path.join(patient_folder, image_file), conf=CONF, verbose=False)[0].boxes.cls

        # Count the number of plasma cells (class 0) and non-plasma cells (class 1).
        plasma_cells += int((detections == 0).sum())
        non_plasma_cells += int((detections == 1).sum())

    # Calculate the total number of cells and the percentage of plasma cells.
    total = plasma_cells + non_plasma_cells    
    percentage = round(plasma_cells / total, 3)

    # Determine the diagnosis based on the percentage of plasma cells.
    # If the percentage is greater than or equal to 10%, classify as 'diseased'; otherwise, as 'normal'.
    diagnosis = 'diseased' if percentage >= 0.1 else 'normal'

    # Append the results for the current patient to the results list.
    results.append([patient, plasma_cells, non_plasma_cells, total, percentage, diagnosis])

100%|███████████████████████████████████████████████████████████████████████████████████| 10/10 [00:19<00:00,  1.90s/it]


In [37]:
# Load and diaply the ground truth data from a CSV file containing patient diagnoses.
ground_truth = pd.read_csv('../data/detection/patients/diagnosis.csv')
ground_truth

,patient,plasma_cells,non_plasma_cells,total,percentage,diagnosis
0,1,82,120,202,0.406,diseased
1,2,78,127,205,0.374,diseased
2,3,72,128,200,0.360,diseased
3,4,82,116,198,0.414,diseased
4,5,71,129,200,0.355,diseased
5,6,15,185,200,0.075,normal
6,7,18,187,205,0.088,normal
7,8,9,199,208,0.043,normal
8,9,14,188,202,0.069,normal
9,10,15,188,203,0.074,normal


In [38]:
# Display the patient diagnoses generated by the model.
results = pd.DataFrame(results, columns=['patient', 'plasma_cells', 'non_plasma_cell', 'total', 'percentage', 'diagnosis'])
results

,patient,plasma_cells,non_plasma_cell,total,percentage,diagnosis
0,1,82,78,160,0.512,diseased
1,2,73,84,157,0.465,diseased
2,3,62,80,142,0.437,diseased
3,4,64,70,134,0.478,diseased
4,5,58,72,130,0.446,diseased
5,6,17,156,173,0.098,normal
6,7,21,165,186,0.113,diseased
7,8,11,186,197,0.056,normal
8,9,20,181,201,0.100,diseased
9,10,13,184,197,0.066,normal


In [33]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
from ultralytics import YOLO

# Path to your validation images and labels
# You need to adapt these paths based on your dataset structure
VAL_IMAGES_DIR = '../diffusion_model/synthetic_original_merged/5fold/fold2/val/images'  # Change to your validation path
VAL_LABELS_DIR = '../diffusion_model/synthetic_original_merged/5fold/fold2/val/labels'  # Change to your validation labels path

# Load your best model (use fold2 since that's what you're using)
model = YOLO('../../../../diffusion_model/synthetic_orignal_merged/5fold/runs/fold2/weights/best.pt')
CONF = 0.8

def get_validation_image_paths(val_dir):
    """Get all validation image paths"""
    if os.path.exists(val_dir):
        return sorted([os.path.join(val_dir, f) for f in os.listdir(val_dir) 
                      if f.endswith(('.jpg', '.png', '.jpeg'))])
    else:
        # Alternative: if your validation data is in patient folders
        print(f"Directory {val_dir} not found. Using patient folders instead...")
        patient_folders = []
        base_dir = '../data/detection/patients'
        for patient in os.listdir(base_dir):
            patient_path = os.path.join(base_dir, patient, 'images')
            if os.path.isdir(patient_path):
                for img in os.listdir(patient_path):
                    patient_folders.append(os.path.join(patient_path, img))
        return sorted(patient_folders)

def count_cells_from_label(label_path, img_shape):
    """Extract ground truth counts from YOLO format label file"""
    if not os.path.exists(label_path):
        return 0, 0
    
    plasma = 0
    non_plasma = 0
    
    with open(label_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 1:
                cls = int(parts[0])
                if cls == 0:
                    plasma += 1
                elif cls == 1:
                    non_plasma += 1
    
    return plasma, non_plasma

def get_predictions(image_path, model, conf=0.8):
    """Get prediction counts from YOLO model"""
    results = model(image_path, conf=conf, verbose=False)[0]
    
    if results.boxes is not None:
        cls = results.boxes.cls.cpu().numpy()
        plasma = int((cls == 0).sum())
        non_plasma = int((cls == 1).sum())
    else:
        plasma = 0
        non_plasma = 0
    
    return plasma, non_plasma

# Get all validation images
validation_images = get_validation_image_paths(VAL_IMAGES_DIR)

# Store comparisons
comparisons = []

print(f"Analyzing {len(validation_images)} validation images...")

for img_path in tqdm(validation_images[:100]):  # Start with 100 images for quick analysis
    # Get predictions
    pred_plasma, pred_non = get_predictions(img_path, model, CONF)
    
    # Get ground truth
    label_path = img_path.replace('images', 'labels').replace('.jpg', '.txt').replace('.png', '.txt')
    
    # For patient folder structure, adjust label path
    if 'patients' in img_path:
        label_path = img_path.replace('/images/', '/labels/').replace('.jpg', '.txt')
    
    if os.path.exists(label_path):
        gt_plasma, gt_non = count_cells_from_label(label_path, None)
        
        # Store for analysis
        comparisons.append({
            'image': os.path.basename(img_path),
            'gt_plasma': gt_plasma,
            'pred_plasma': pred_plasma,
            'gt_non': gt_non,
            'pred_non': pred_non,
            'gt_total': gt_plasma + gt_non,
            'pred_total': pred_plasma + pred_non
        })

# Create DataFrame
df_comparisons = pd.DataFrame(comparisons)

# Calculate ratios
df_comparisons['plasma_ratio'] = df_comparisons['pred_plasma'] / df_comparisons['gt_plasma'].replace(0, 1)
df_comparisons['non_plasma_ratio'] = df_comparisons['pred_non'] / df_comparisons['gt_non'].replace(0, 1)
df_comparisons['total_ratio'] = df_comparisons['pred_total'] / df_comparisons['gt_total'].replace(0, 1)

# Calculate correction factors (avoid infinite values)
plasma_ratios = df_comparisons[df_comparisons['gt_plasma'] > 0]['plasma_ratio']
non_plasma_ratios = df_comparisons[df_comparisons['gt_non'] > 0]['non_plasma_ratio']

# Median ratios are more robust to outliers
median_plasma_ratio = np.median(plasma_ratios)
median_non_plasma_ratio = np.median(non_plasma_ratios)

print("\n" + "="*50)
print("CALIBRATION ANALYSIS")
print("="*50)
print(f"Plasma cell detection ratio: {median_plasma_ratio:.3f}")
print(f"Non-plasma cell detection ratio: {median_non_plasma_ratio:.3f}")
print(f"\nRecommended Calibration Factors:")
print(f"PLASMA_CALIBRATION = {1/median_plasma_ratio:.3f}")
print(f"NON_PLASMA_CALIBRATION = {1/median_non_plasma_ratio:.3f}")

# Display statistics
print("\n" + "="*50)
print("DETAILED STATISTICS")
print("="*50)
print(df_comparisons.describe())

# Show sample comparisons
print("\n" + "="*50)
print("SAMPLE COMPARISONS (First 10 images)")
print("="*50)
print(df_comparisons[['image', 'gt_plasma', 'pred_plasma', 'gt_non', 'pred_non']].head(10))

Directory ../diffusion_model/synthetic_original_merged/5fold/fold2/val/images not found. Using patient folders instead...
Analyzing 344 validation images...


100%|█████████████████████████████████████████████████████████████████████████████████| 100/100 [00:06<00:00, 15.64it/s]


CALIBRATION ANALYSIS
Plasma cell detection ratio: 1.000
Non-plasma cell detection ratio: 0.667

Recommended Calibration Factors:
PLASMA_CALIBRATION = 1.000
NON_PLASMA_CALIBRATION = 1.500

DETAILED STATISTICS
        gt_plasma  pred_plasma      gt_non    pred_non   gt_total  pred_total  \
count  100.000000   100.000000  100.000000  100.000000  100.00000  100.000000   
mean     2.180000     2.050000    3.530000    2.330000    5.71000    4.380000   
std      2.622282     2.157323    2.240378    2.020376    3.57967    2.612248   
min      0.000000     0.000000    0.000000    0.000000    1.00000    0.000000   
25%      1.000000     1.000000    2.000000    1.000000    3.00000    3.000000   
50%      1.500000     2.000000    3.000000    2.000000    5.00000    4.000000   
75%      3.000000     3.000000    5.000000    3.000000    7.00000    6.000000   
max     18.000000    12.000000   10.000000   10.000000   25.00000   14.000000   

       plasma_ratio  non_plasma_ratio  total_ratio  
count   

In [34]:
import pandas as pd
from tqdm import tqdm
from ultralytics import YOLO

# Load your model
model = YOLO('../../../../diffusion_model/synthetic_orignal_merged/5fold/runs/fold2/weights/best.pt')
CONF = 0.8

# Calibration factors from your validation analysis
PLASMA_CALIBRATION = 1.000  # No adjustment needed
NON_PLASMA_CALIBRATION = 1.500  # Multiply by 1.5 to recover missed cells

# Initialize results list
results_calibrated = []

print("Analyzing patient folders with calibration...")

for patient_folder in tqdm(patient_folders):
    # Get list of image files
    image_files = os.listdir(patient_folder)
    
    # Extract patient identifier
    patient = int(patient_folder.split('/')[-2].split(' ')[-1])
    
    # Initialize counters
    plasma_cells_raw = 0
    non_plasma_cells_raw = 0
    
    # Process each image
    for image_file in image_files:
        detections = model(os.path.join(patient_folder, image_file), conf=CONF, verbose=False)[0].boxes.cls
        plasma_cells_raw += int((detections == 0).sum())
        non_plasma_cells_raw += int((detections == 1).sum())
    
    # Apply calibration
    plasma_cells_calibrated = int(plasma_cells_raw * PLASMA_CALIBRATION)
    non_plasma_cells_calibrated = int(non_plasma_cells_raw * NON_PLASMA_CALIBRATION)
    
    # Calculate totals and percentage
    total_calibrated = plasma_cells_calibrated + non_plasma_cells_calibrated
    percentage_calibrated = round(plasma_cells_calibrated / total_calibrated, 3) if total_calibrated > 0 else 0
    
    # Diagnosis (using 10% threshold)
    diagnosis = 'diseased' if percentage_calibrated >= 0.1 else 'normal'
    
    # Store results
    results_calibrated.append([
        patient, 
        plasma_cells_calibrated, 
        non_plasma_cells_calibrated, 
        total_calibrated, 
        percentage_calibrated, 
        diagnosis
    ])

# Create DataFrame
results_df = pd.DataFrame(results_calibrated, columns=['patient', 'plasma_cells', 'non_plasma_cell', 'total', 'percentage', 'diagnosis'])

print("\n" + "="*70)
print("CALIBRATED PATIENT RESULTS")
print("="*70)
print(results_df)

# Compare with ground truth
ground_truth = pd.read_csv('../data/detection/patients/diagnosis.csv')

print("\n" + "="*70)
print("COMPARISON WITH GROUND TRUTH")
print("="*70)

# Merge predictions with ground truth
comparison = results_df.merge(ground_truth, on='patient', suffixes=('_pred', '_gt'))

# Calculate accuracy
comparison['correct'] = comparison['diagnosis_pred'] == comparison['diagnosis_gt']
accuracy = comparison['correct'].mean()

print(f"\nOverall Accuracy: {accuracy:.2%}")
print(f"Correct predictions: {comparison['correct'].sum()}/{len(comparison)}")

# Show detailed comparison
print("\n" + "-"*70)
print("Patient-by-Patient Comparison:")
print("-"*70)

for _, row in comparison.iterrows():
    status = "✓" if row['correct'] else "✗"
    print(f"Patient {int(row['patient'])}: "
          f"GT: {row['plasma_cells_gt']}/{row['total_gt']} ({row['percentage_gt']:.1%}) {row['diagnosis_gt']} → "
          f"Pred: {row['plasma_cells_pred']}/{row['total_pred']} ({row['percentage_pred']:.1%}) {row['diagnosis_pred']} {status}")

# Show the improvement
print("\n" + "="*70)
print("IMPROVEMENT ANALYSIS")
print("="*70)

# Original results (from your earlier run with fold2 at CONF=0.8)
original_results = pd.DataFrame([
    [1, 82, 78, 160, 0.512, 'diseased'],
    [2, 73, 84, 157, 0.465, 'diseased'],
    [3, 62, 80, 142, 0.437, 'diseased'],
    [4, 64, 70, 134, 0.478, 'diseased'],
    [5, 58, 72, 130, 0.446, 'diseased'],
    [6, 17, 156, 173, 0.098, 'normal'],
    [7, 21, 165, 186, 0.113, 'diseased'],
    [8, 11, 186, 197, 0.056, 'normal'],
    [9, 20, 181, 201, 0.100, 'diseased'],
    [10, 13, 184, 197, 0.066, 'normal']
], columns=['patient', 'plasma_cells', 'non_plasma_cell', 'total', 'percentage', 'diagnosis'])

# Compare
for patient in [6, 7, 8, 9, 10]:
    orig = original_results[original_results['patient'] == patient].iloc[0]
    calib = results_df[results_df['patient'] == patient].iloc[0]
    gt = ground_truth[ground_truth['patient'] == patient].iloc[0]
    
    print(f"\nPatient {patient}:")
    print(f"  GT: {gt['percentage']:.1%} → {gt['diagnosis']}")
    print(f"  Original: {orig['percentage']:.1%} → {orig['diagnosis']}")
    print(f"  Calibrated: {calib['percentage']:.1%} → {calib['diagnosis']}")

Analyzing patient folders with calibration...


100%|███████████████████████████████████████████████████████████████████████████████████| 10/10 [00:17<00:00,  1.75s/it]


CALIBRATED PATIENT RESULTS
   patient  plasma_cells  non_plasma_cell  total  percentage diagnosis
0        1            82              117    199       0.412  diseased
1        2            73              126    199       0.367  diseased
2        3            62              120    182       0.341  diseased
3        4            64              105    169       0.379  diseased
4        5            58              108    166       0.349  diseased
5        6            17              234    251       0.068    normal
6        7            21              247    268       0.078    normal
7        8            11              279    290       0.038    normal
8        9            20              271    291       0.069    normal
9       10            13              276    289       0.045    normal

COMPARISON WITH GROUND TRUTH

Overall Accuracy: 100.00%
Correct predictions: 10/10

----------------------------------------------------------------------
Patient-by-Patient Comparison:
----